# MAPPO Padded Source Curriculum 50x50

Warm-start the completed exploration-to-forage `20x20` stage inside a `50x50` hidden arena. Random hub and food placement stay in the interior `20x20` window, while the curriculum decreases the number of food-source positions.

In [ ]:
from pathlib import Path
import os
import sys

# Set these before importing JAX in this kernel.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.35")
if "jax" in sys.modules:
    print("Restart the kernel before rerunning training; JAX was already imported.")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from ant_byte_env import notebook_workflows as workflows

runtime_status = workflows.configure_jax_notebook_runtime()
workflows.assert_notebook_resources_available(runtime_status)
runtime_status

In [ ]:
import importlib

import jax

from ant_byte_env.training.jax_mappo import runner as jax_runner

workflows = importlib.reload(workflows)
jax_runner = importlib.reload(jax_runner)
print(f"JAX device: {jax.devices()[0]}")

## Quick Smoke Run

Run one tiny job before loading the warm-start checkpoint.

In [ ]:
smoke_metrics = workflows.run_jax_smoke(jax_runner.main)
smoke_metrics

## Curriculum Settings

Edit `experiments/exploration_to_forage_padded_sources_50x50.json` for durable source-count, margin, reward-scale, or budget changes.

In [ ]:
EXPERIMENT_CONFIG = PROJECT_ROOT / "experiments" / "exploration_to_forage_padded_sources_50x50.json"
experiment = workflows.load_jax_experiment(EXPERIMENT_CONFIG)
experiment_args = dict(experiment.args)

RUN_DIR = PROJECT_ROOT / "runs" / "notebooks" / "exploration_to_forage_padded_sources_50x50"
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
MEDIA_DIR = RUN_DIR / "media"
ROLLOUT_TILE_SIZE = workflows.NOTEBOOK_ROLLOUT_TILE_SIZE
ROLLOUT_POLICY_TEMPERATURE = workflows.notebook_rollout_policy_temperature(experiment.metadata)
WANDB_VIDEO_MAX_FRAMES = int(experiment.metadata["wandb_video_max_frames"])
WANDB_VIDEO_STAGE_NAMES = tuple(experiment.metadata["wandb_preview_stage_names"])
WANDB_VIDEO_ROLLOUT_COUNT = int(experiment.metadata.get("wandb_preview_rollout_count", 1))
STAGE_UPDATE_MULTIPLIER = float(experiment.metadata.get("stage_update_multiplier", 1.0))
SOURCE_CHECKPOINT = workflows.resolve_project_path(PROJECT_ROOT, experiment_args["load_model"])
BEST_CHECKPOINT_PATH = workflows.resolve_project_path(PROJECT_ROOT, experiment_args["save_best_model"])
if not SOURCE_CHECKPOINT.exists():
    raise FileNotFoundError(f"Run or restore the source checkpoint first: {SOURCE_CHECKPOINT}")
experiment_args["load_model"] = str(SOURCE_CHECKPOINT)
experiment_args["save_best_model"] = str(BEST_CHECKPOINT_PATH)

SOURCE_COUNTS = tuple(int(count) for count in experiment.metadata["food_source_counts"])
CURRICULUM_STAGES = workflows.build_food_source_curriculum_stages(
    experiment_args,
    source_counts=SOURCE_COUNTS,
    visit_reward_schedule=experiment.metadata.get("visit_reward_schedule"),
    stage_update_multiplier=STAGE_UPDATE_MULTIPLIER,
)
GLOBAL_UPDATE_CAP = max(int(stage["global_update_cap"]) for stage in CURRICULUM_STAGES)
UPDATE_TIMESTEPS = workflows.update_timesteps(
    num_envs=int(experiment_args["num_envs"]),
    num_steps=int(experiment_args["num_steps"]),
)
WANDB_PROJECT = "cool-antz"
WANDB_ENTITY = None
WANDB_GROUP = "exploration_to_forage_padded_sources_50x50"
WANDB_RUN_NAME = WANDB_GROUP
WANDB_MODE = "online"
COMMON_ARGS = workflows.config_common_args(
    experiment_args,
    exclude=workflows.EXPLORATION_TO_FORAGE_ARG_EXCLUDES,
)
{
    "source_checkpoint": SOURCE_CHECKPOINT,
    "best_checkpoint": BEST_CHECKPOINT_PATH,
    "layout_margin": experiment_args.get("layout_margin"),
    "food_source_counts": [stage["food_sources"] for stage in CURRICULUM_STAGES],
    "stage_training_profiles": [
        (stage["name"], stage["food_sources"], stage["global_update_cap"], stage["num_steps"], stage["gamma"])
        for stage in CURRICULUM_STAGES
    ],
    "total_updates_per_stage": GLOBAL_UPDATE_CAP,
    "update_timesteps": UPDATE_TIMESTEPS,
}

## Train Source Curriculum

In [ ]:
training_result = workflows.run_forage_curriculum(
    stages=CURRICULUM_STAGES,
    checkpoint_dir=CHECKPOINT_DIR,
    common_args=COMMON_ARGS,
    update_timesteps_per_stage=UPDATE_TIMESTEPS,
    global_update_cap=GLOBAL_UPDATE_CAP,
    train_main=jax_runner.main,
    initial_checkpoint=SOURCE_CHECKPOINT,
    wandb_project=WANDB_PROJECT,
    wandb_entity=WANDB_ENTITY,
    wandb_group=WANDB_GROUP,
    wandb_run_name=WANDB_RUN_NAME,
    wandb_mode=WANDB_MODE,
    wandb_tags=["exploration-to-forage", "padded-arena", "food-source-curriculum", "50x50"],
    wandb_notes=experiment.metadata["notes"],
    wandb_artifact_paths=[EXPERIMENT_CONFIG],
    wandb_artifact_prefix="exploration-to-forage-padded-sources",
    checkpoint_name_prefix="jax_mappo_exploration_to_forage_padded_sources",
    wandb_video_key_prefix="videos/exploration_to_forage/padded_sources",
    wandb_video_max_frames=WANDB_VIDEO_MAX_FRAMES,
    wandb_video_stage_names=WANDB_VIDEO_STAGE_NAMES,
    wandb_video_policy_temperature=ROLLOUT_POLICY_TEMPERATURE,
    wandb_video_rollout_count=WANDB_VIDEO_ROLLOUT_COUNT,
)
FINAL_CHECKPOINT_PATH = training_result["final_checkpoint_path"]
ROLLOUT_CHECKPOINT_PATH = FINAL_CHECKPOINT_PATH
training_result

## Optional Local Render and Vault

In [ ]:
# rollout_result = workflows.render_jax_checkpoint_rollout(
#     run_dir=RUN_DIR,
#     checkpoint_path=ROLLOUT_CHECKPOINT_PATH,
#     media_dir=MEDIA_DIR,
#     rollout_filename="jax_mappo_exploration_to_forage_padded_sources_best_rollout.mp4",
#     title="JAX MAPPO padded source curriculum rollout",
#     description="Deterministic rollout from the eval-selected padded source curriculum checkpoint.",
#     metadata={
#         "experiment_config": str(EXPERIMENT_CONFIG),
#         "source_checkpoint": str(SOURCE_CHECKPOINT),
#         "best_checkpoint": str(BEST_CHECKPOINT_PATH),
#         "layout_margin": experiment_args.get("layout_margin"),
#         "food_source_counts": [stage["food_sources"] for stage in CURRICULUM_STAGES],
#     },
#     tile_size=ROLLOUT_TILE_SIZE,
#     policy_temperature=ROLLOUT_POLICY_TEMPERATURE,
#     wandb_project=WANDB_PROJECT,
#     wandb_entity=WANDB_ENTITY,
#     wandb_group=WANDB_GROUP,
#     wandb_run_name=f"{WANDB_GROUP}_rollout",
#     wandb_mode="disabled",
#     wandb_video_key=None,
#     wandb_step=training_result["stage_metrics"][-1].get("curriculum_global_step"),
# )
# rollout_result